# RoBERTa-base — Treino com Optuna

Fine-tuning de `roberta-base` para classificação multi-classe (5 classes: Google, Anthropic, Meta, OpenAI, Human).

`roberta-base` (modelo completo, ~125M params).

Usa **Optuna** para otimização de hiperparâmetros e valida com o `dataset-exemplos.csv`.

In [1]:
import sys
print('Python:', sys.version)
print('Executável:', sys.executable)

import numpy as np
import pandas as pd
import os, copy
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
import transformers
print('transformers:', transformers.__version__)
print('torch:', torch.__version__)

from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import optuna

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID  = {'google': 0, 'anthropic': 1, 'meta': 2, 'openai': 3, 'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
N_CLASSES = 5

Python: 3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Executável: c:\Users\gbarr\Desktop\Universidade\Mestrado\1Ano2Semestre\Aprendizagem Profunda\TP-AP2026\ap\Scripts\python.exe
transformers: 5.3.0
torch: 2.6.0+cu124
device: cuda


## 1. Carregar dados

In [2]:
df_full  = pd.read_csv('../datasets/dataset_v2_full.csv',    sep=';')
df_ex    = pd.read_csv('../datasets/dataset-exemplos.csv',   sep=';')
df_subm1 = pd.read_csv('../Subm1/subm1_labels_revealed.csv', sep=';')

def load_xy(df):
    # remover linhas sem Label válida
    df = df.dropna(subset=['Label'])
    df = df[df['Label'].str.strip().str.lower().isin(LABEL2ID.keys())]
    texts  = df['Text'].fillna('').tolist()
    labels = [LABEL2ID[l.strip().lower()] for l in df['Label'].tolist()]
    return texts, labels

texts_synth, y_synth = load_xy(df_full)
texts_real,  y_real  = load_xy(df_subm1)
texts_val,   y_val   = load_xy(df_ex)

# upsample dados reais do docente (mais representativos do teste)
REAL_WEIGHT = 15
texts_train = texts_synth + texts_real * REAL_WEIGHT
y_train     = y_synth     + y_real     * REAL_WEIGHT

# class weights baseados nos dados reais (que refletem a distribuição do teste)
cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_real + y_val)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

print(f'treino    : {len(texts_train)} ({len(texts_synth)} sintéticos + {len(texts_real)}×{REAL_WEIGHT} reais)')
print(f'validação : {len(texts_val)} exemplos reais do docente')
print('class weights:', {ID2LABEL[i]: f'{w:.2f}' for i, w in enumerate(cw)})

# distribuição das classes no treino
from collections import Counter
dist = Counter(y_train)
print('distribuição treino:', {ID2LABEL[k]: v for k, v in sorted(dist.items())})

treino    : 6500 (5000 sintéticos + 100×15 reais)
validação : 125 exemplos reais do docente
class weights: {'google': '1.36', 'anthropic': '1.12', 'meta': '1.29', 'openai': '1.45', 'human': '0.52'}
distribuição treino: {'google': 1255, 'anthropic': 1255, 'meta': 1270, 'openai': 1210, 'human': 1510}


## 2. Funções auxiliares (Dataset, avaliação, loss)

In [3]:
MODEL_NAME = 'roberta-base'
MAX_LEN    = 128

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

def evaluate(model, loader):
    model.eval()
    preds_all, true_all = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            preds_all.extend(torch.argmax(out.logits, dim=1).cpu().tolist())
            true_all.extend(batch['labels'].tolist())
    acc = sum(p == t for p, t in zip(preds_all, true_all)) / len(true_all)
    f1  = f1_score(true_all, preds_all, average='macro')
    return acc, f1, preds_all

def label_smoothing_loss(logits, labels, n_classes, smoothing=0.1, weights=None):
    log_probs = F.log_softmax(logits, dim=-1)
    smooth_target = torch.full_like(log_probs, smoothing / (n_classes - 1))
    smooth_target.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
    loss = -(smooth_target * log_probs).sum(dim=-1)
    if weights is not None:
        loss = loss * weights[labels]
    return loss.mean()

print(f'modelo base: {MODEL_NAME}')
print('tokenizer carregado')

modelo base: roberta-base
tokenizer carregado


## 3. Otimização de hiperparâmetros com Optuna

Espaço de busca:
- **learning_rate**: 1e-5 a 5e-5
- **batch_size**: 16 ou 32
- **label_smoothing**: 0.05 a 0.2
- **warmup_fraction**: 0.05 a 0.2
- **weight_decay**: 0.001 a 0.1
- **real_weight**: 5 a 20 (upsampling dos dados reais)

Cada trial treina por no máximo 10 épocas com early stopping (patience=3).
O objetivo é maximizar o **F1-macro** no `dataset-exemplos.csv`.

In [4]:
# pré-tokenizar validação uma só vez (não depende dos hiperparâmetros)
val_ds     = TextDataset(texts_val, y_val, tokenizer, MAX_LEN)
val_loader = DataLoader(val_ds, batch_size=64)
print(f'validação tokenizada: {len(val_ds)} amostras')


def train_one_trial(trial):
    """Treina o modelo com os hiperparâmetros sugeridos pelo Optuna."""

    # --- hiperparâmetros a otimizar ---
    lr            = trial.suggest_float('lr', 1e-5, 5e-5, log=True)
    batch_size    = trial.suggest_categorical('batch_size', [16, 32])
    smoothing     = trial.suggest_float('label_smoothing', 0.05, 0.2)
    warmup_frac   = trial.suggest_float('warmup_frac', 0.05, 0.2)
    weight_decay  = trial.suggest_float('weight_decay', 1e-3, 0.1, log=True)
    real_wt       = trial.suggest_int('real_weight', 5, 20)

    MAX_EPOCHS = 10
    PATIENCE   = 3

    # reconstruir treino com o real_weight desta trial
    t_train = texts_synth + texts_real * real_wt
    y_tr    = y_synth     + y_real     * real_wt
    train_ds     = TextDataset(t_train, y_tr, tokenizer, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    # modelo fresco
    model = RobertaForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=N_CLASSES,
        id2label=ID2LABEL, label2id=LABEL2ID
    ).to(device)

    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps  = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(total_steps * warmup_frac)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

    best_f1    = 0.0
    no_improve = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            ids    = batch['input_ids'].to(device)
            mask   = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
                logits = model(input_ids=ids, attention_mask=mask).logits
                loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                              smoothing=smoothing, weights=class_weights)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        val_acc, val_f1, _ = evaluate(model, val_loader)

        # reportar ao Optuna para pruning
        trial.report(val_f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_f1 > best_f1:
            best_f1    = val_f1
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                break

    # limpar memória GPU
    del model, optimizer, scheduler, scaler, train_ds, train_loader
    torch.cuda.empty_cache()

    return best_f1


print('função de treino definida')

validação tokenizada: 125 amostras
função de treino definida


In [5]:
# ── Executar a otimização Optuna ──────────────────────────────────────────────
# Ajustar n_trials conforme o tempo disponível:
#   - 5 trials  ≈ 30-40 min (com GPU)
#   - 10 trials ≈ 1-1.5 h
#   - 15 trials ≈ 2 h

N_TRIALS = 10

study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=3),
    study_name='deberta-v3-hparam-search'
)
study.optimize(train_one_trial, n_trials=N_TRIALS, show_progress_bar=True)

print('\n' + '='*60)
print(f'Melhor F1-macro: {study.best_value:.4f}')
print('Melhores hiperparâmetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

[I 2026-03-24 14:33:56,313] A new study created in memory with name: deberta-v3-hparam-search


  0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[I 2026-03-24 14:46:07,354] Trial 0 finished with value: 0.661243030141726 and parameters: {'lr': 1.124635937951412e-05, 'batch_size': 32, 'label_smoothing': 0.08503250640629456, 'warmup_frac': 0.09573596487342845, 'weight_decay': 0.0019686594048313262, 'real_weight': 16}. Best is trial 0 with value: 0.661243030141726.


...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[I 2026-03-24 16:37:57,724] Trial 8 finished with value: 0.7028942138891612 and parameters: {'lr': 4.679196971838145e-05, 'batch_size': 16, 'label_smoothing': 0.088760656359572, 'warmup_frac': 0.0874559848436893, 'weight_decay': 0.0039798759624862165, 'real_weight': 20}. Best is trial 6 with value: 0.7452064861410655.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[I 2026-03-24 16:48:49,202] Trial 9 pruned. 

Melhor F1-macro: 0.7452
Melhores hiperparâmetros:
  lr: 1.1027749296100164e-05
  batch_size: 32
  label_smoothing: 0.06168830566039848
  warmup_frac: 0.16159961620711
  weight_decay: 0.015107069219545775
  real_weight: 9


## 4. Treino final com os melhores hiperparâmetros

Retreinar o modelo com os melhores hiperparâmetros encontrados, desta vez com mais épocas e guardando o melhor checkpoint.

In [6]:
# ── Treino final com melhores hiperparâmetros ─────────────────────────────────
bp = study.best_params

LR           = bp['lr']
BATCH_SIZE   = bp['batch_size']
LABEL_SMOOTH = bp['label_smoothing']
WARMUP_FRAC  = bp['warmup_frac']
WEIGHT_DECAY = bp['weight_decay']
REAL_WT      = bp['real_weight']
MAX_EPOCHS   = 20   # mais épocas para o treino final
PATIENCE     = 5

# reconstruir dados de treino
t_train_final = texts_synth + texts_real * REAL_WT
y_train_final = y_synth     + y_real     * REAL_WT

train_ds     = TextDataset(t_train_final, y_train_final, tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print(f'Treino final: {len(train_ds)} amostras, batch_size={BATCH_SIZE}')
print(f'Hiperparâmetros: lr={LR:.2e}, smoothing={LABEL_SMOOTH:.3f}, '
      f'warmup={WARMUP_FRAC:.3f}, wd={WEIGHT_DECAY:.4f}, real_wt={REAL_WT}')

Treino final: 5900 amostras, batch_size=32
Hiperparâmetros: lr=1.10e-05, smoothing=0.062, warmup=0.162, wd=0.0151, real_wt=9


In [7]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label=ID2LABEL, label2id=LABEL2ID
).to(device)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

best_f1    = 0.0
best_acc   = 0.0
best_state = None
no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                          smoothing=LABEL_SMOOTH, weights=class_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    val_acc, val_f1, _ = evaluate(model, val_loader)
    marker = ' *' if val_f1 > best_f1 else ''
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | loss={total_loss/len(train_loader):.4f} '
          f'| val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_acc   = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping na época {epoch}')
            break

model.load_state_dict(best_state)
print(f'\nMelhor modelo: val_acc={best_acc:.4f}  val_f1={best_f1:.4f}')

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 01/20 | loss=1.7681 | val_acc=0.1280 | val_f1=0.0457 *
Epoch 02/20 | loss=0.9619 | val_acc=0.5200 | val_f1=0.4552 *
Epoch 03/20 | loss=0.4183 | val_acc=0.7680 | val_f1=0.7028 *
Epoch 04/20 | loss=0.3694 | val_acc=0.7440 | val_f1=0.6655
Epoch 05/20 | loss=0.3614 | val_acc=0.7680 | val_f1=0.6922
Epoch 06/20 | loss=0.3605 | val_acc=0.7520 | val_f1=0.6624
Epoch 07/20 | loss=0.3603 | val_acc=0.7520 | val_f1=0.6639
Epoch 08/20 | loss=0.3604 | val_acc=0.7520 | val_f1=0.6639
Early stopping na época 8

Melhor modelo: val_acc=0.7680  val_f1=0.7028


## 5. Guardar modelo e avaliar

In [8]:
SAVE_DIR = '../models/model_roberta'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'modelo guardado em {SAVE_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

modelo guardado em ../models/model_roberta


In [ ]:
val_acc, val_f1, val_preds = evaluate(model, val_loader)
print(f'[dataset-exemplos] accuracy={val_acc:.4f}  f1-macro={val_f1:.4f}')
print()
print(classification_report(
    [ID2LABEL[l] for l in y_val],
    [ID2LABEL[p] for p in val_preds],
    digits=3
))

## 6. Resumo dos resultados do Optuna

In [9]:
# tabela com todas as trials
trials_df = study.trials_dataframe()
trials_df = trials_df.sort_values('value', ascending=False)
print('Top 5 trials:')
cols = ['number', 'value', 'params_lr', 'params_batch_size',
        'params_label_smoothing', 'params_warmup_frac',
        'params_weight_decay', 'params_real_weight']
display_cols = [c for c in cols if c in trials_df.columns]
print(trials_df[display_cols].head(5).to_string(index=False))

Top 5 trials:
 number    value  params_lr  params_batch_size  params_label_smoothing  params_warmup_frac  params_weight_decay  params_real_weight
      6 0.745206   0.000011                 32                0.061688            0.161600             0.015107                   9
      3 0.717346   0.000014                 32                0.061380            0.184211             0.003074                  13
      1 0.710573   0.000012                 32                0.165745            0.185895             0.003963                  10
      8 0.702894   0.000047                 16                0.088761            0.087456             0.003980                  20
      4 0.693189   0.000022                 32                0.126330            0.146461             0.025498                  11
